# Laboratorio #3

* Josue Say - 228801
* Flavio Galán - 22386

## Repositorio

- [Enlace](https://github.com/JosueSay/labs-ds/tree/main/lab4)
- [Data](https://drive.google.com/file/d/1HtrCx-AEMuC6CeKCLVbERCJMxmqiPB6v/view?usp=sharing)

## Librerías y constantes

In [195]:
import csv
import folium
import json
import math
import os
import time
import warnings
from datetime import datetime, timedelta
from pathlib import Path
import branca.colormap as bcm
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import openeo
import rasterio
from dotenv import load_dotenv
from rasterio.crs import CRS as RIO_CRS
from rasterio.mask import mask as rio_mask
from rasterio.transform import from_bounds, array_bounds
from rasterio.warp import reproject, Resampling, transform_bounds
from shapely.geometry import mapping
from sentinelhub import (
    BBox,
    CRS,
    DataCollection,
    MimeType,
    SHConfig,
    SentinelHubRequest,
    bbox_to_dimensions,
)
from sentinelhub.exceptions import DownloadFailedException, SHRateLimitWarning

warnings.filterwarnings("ignore", category=SHRateLimitWarning)

# !pip install -r requirements.txt
load_dotenv() 

True

In [175]:
# === GLOBAL ===
BASE_DIR = Path.cwd()
DATA_NAME = "data"
DATA_DIR = BASE_DIR / DATA_NAME
DO_PIPELINE = True

# === SECTION 1 ===
# Fechas con nubosidad <20%
PROF_DATES = [
    "2025-02-07","2025-02-10","2025-02-25","2025-02-27",
    "2025-03-02","2025-03-04","2025-03-07","2025-03-09","2025-03-12","2025-03-14","2025-03-19","2025-03-22","2025-03-24","2025-03-26",
    "2025-04-03","2025-04-11","2025-04-13","2025-04-15","2025-04-16","2025-04-18","2025-04-28",
    "2025-05-03","2025-05-13","2025-05-28",
    "2025-07-10","2025-07-17","2025-07-20","2025-07-24","2025-08-01",
]

USE_SUBSET = False        # ¿Usar un subconjunto (2 fechas por mes) en vez de todas?
SUBSET_MONTHS = [6, 7]    # junio, julio
PER_MONTH = 2             # 2 fechas por mes

# === SECTION 2 ===
MAX_CLOUD = 20
RESOLUTION = 20  # metros
DO_RGB_PREVIEW = False  # True para un GTIFF RGB por lago

BANDS_RGB   = ["B04","B03","B02"]
BANDS_INDEX = ["B02","B03","B04","B05","B08","B8A","B11","B12"]  # para NDVI/NDWI/Cyano

# === SECTION 3 ===
OUT_SUBDIR  = "Cyano_SH"

# Ritmo de descarga
PAUSE_BETWEEN_REQUESTS = 2.0   # seg, pausa fija entre fechas
MAX_RETRIES            = 5     # reintentos ante 429/limit
BASE_BACKOFF_SEC       = 5.0   # backoff exponencial: 5, 10, 20, ...

EVALSCRIPT_CHL = """
function setup() {
  return {
    input: [{ bands: ["B04","B05","SCL","dataMask"] }],
    output: { bands: 1, sampleType: "FLOAT32" }
  };
}

function isCloudOrShadow(scl) {
  // S2 L2A SCL: 3=Cloud Shadows, 8=Cloud (med), 9=Cloud (high), 10=Thin cirrus, 11=Snow/Ice
  return (scl === 3 || scl === 8 || scl === 9 || scl === 10 || scl === 11);
}

function evaluatePixel(s) {
  if (s.dataMask === 0) return [NaN];
  if (isCloudOrShadow(s.SCL)) return [NaN];

  var ndci = (s.B05 - s.B04) / (s.B05 + s.B04);
  if (!isFinite(ndci)) return [NaN];

  var chl = 826.57*Math.pow(ndci,3) - 176.43*Math.pow(ndci,2) + 19*ndci + 4.071;
  if (!isFinite(chl)) return [NaN];

  return [chl];
}
"""

# === SECTION 4 ===
DIRS = {
    ("Atitlan", "NDVI"): DATA_DIR / "NDVI_Atitlan",
    ("Amatitlan", "NDVI"): DATA_DIR / "NDVI_Amatitlan",
    ("Atitlan", "NDWI"): DATA_DIR / "NDWI_Atitlan",
    ("Amatitlan", "NDWI"): DATA_DIR / "NDWI_Amatitlan",    
    ("Atitlan", "CYANO"): DATA_DIR / "Atitlan" / "Cyano_SH",
    ("Amatitlan", "CYANO"): DATA_DIR / "Amatitlan" / "Cyano_SH",
}

OUT_DIR = DATA_DIR / "export_consolidated"
OUT_DIR.mkdir(parents=True, exist_ok=True)
INTERVALS_PATH = DATA_DIR / "intervals.json"


# === SECTION 5 ===

EXPORT_DIR = DATA_DIR / "export_consolidated"
OUT_DIR2 = DATA_DIR / "analysis"
OUT_DIR2.mkdir(parents=True, exist_ok=True)

LAKES = {
    "Atitlan":   DATA_DIR / "Lago_Atitlan.geojson",
    "Amatitlan": DATA_DIR / "Lago_Amatitlan.geojson",
}

STACKS = {
    ("Atitlan","CYANO"):     (EXPORT_DIR / "Atitlan_CYANO_stack.tif",     EXPORT_DIR / "Atitlan_CYANO_bands_dates.csv"),
    ("Atitlan","NDVI"):      (EXPORT_DIR / "Atitlan_NDVI_stack.tif",      EXPORT_DIR / "Atitlan_NDVI_bands_dates.csv"),
    ("Atitlan","NDWI"):      (EXPORT_DIR / "Atitlan_NDWI_stack.tif",      EXPORT_DIR / "Atitlan_NDWI_bands_dates.csv"),
    ("Amatitlan","CYANO"):   (EXPORT_DIR / "Amatitlan_CYANO_stack.tif",   EXPORT_DIR / "Amatitlan_CYANO_bands_dates.csv"),
    ("Amatitlan","NDVI"):    (EXPORT_DIR / "Amatitlan_NDVI_stack.tif",    EXPORT_DIR / "Amatitlan_NDVI_bands_dates.csv"),
    ("Amatitlan","NDWI"):    (EXPORT_DIR / "Amatitlan_NDWI_stack.tif",    EXPORT_DIR / "Amatitlan_NDWI_bands_dates.csv"),
}


PEAK_METHOD = "percentile"
PEAK_PERCENTILE = 90
ZSCORE_THRESHOLD = 1.0
CYANO_METRIC = "p90"   # puede ser "mean"
CYANO_THRESH = 0.1

# === SECTION 6 ===
EXPORT_DIR2 = DATA_DIR / "export_consolidated"
STACKS2 = {
    ("Atitlan", "CYANO"):   (EXPORT_DIR2 / "Atitlan_CYANO_stack.tif",   EXPORT_DIR2 / "Atitlan_CYANO_bands_dates.csv"),
    ("Amatitlan", "CYANO"): (EXPORT_DIR2 / "Amatitlan_CYANO_stack.tif", EXPORT_DIR2 / "Amatitlan_CYANO_bands_dates.csv"),
}
OUT_DIR3 = DATA_DIR / "maps"
OUT_DIR3.mkdir(parents=True, exist_ok=True)


# === SECTION 7 ===
ANLYSIS = DATA_DIR / "analysis"
OUT_REPORT  = DATA_DIR / "report_outputs"
OUT_REPORT.mkdir(parents=True, exist_ok=True)
LAKES_N = ["Atitlan", "Amatitlan"]
INDICES = ["CYANO", "NDVI", "NDWI"]

## Fechas y nubosidad (inciso 3 -> p1)

**Objetivo:**

* Definir el rango que cubra el período de estudio (p.ej., feb–ago 2025).
* Usar las fechas de la profesora (nubosidad <20%).
* Armar “ventanas” de 1 día para mosaicos diarios (o 2 fechas/mes si así lo pide).

**Por qué:**

* Garantiza calidad (menos nubes) y uniformidad temporal para el análisis.

**Resultado:**

* Un conjunto de **fechas objetivo** sobre las que harás mosaicos/estadísticos.

In [176]:
def buildDailyIntervals(date_list):
    """Devuelve [[startZ, endZ], ...] con ventanas de 1 día."""
    intervals = []
    for d in date_list:
        dt = datetime.fromisoformat(d)
        start = dt.isoformat() + "Z"
        end = (dt + timedelta(days=1)).isoformat() + "Z"
        intervals.append([start, end])
    return intervals

In [177]:
def pickDatesPerMonth(date_list, months, per_month=2):
    """
    Toma 'per_month' fechas por cada mes de 'months' manteniendo orden.
    months: lista de enteros (1-12)
    """
    picked = []
    counters = {m: 0 for m in months}
    for d in date_list:
        m = datetime.fromisoformat(d).month
        if m in counters and counters[m] < per_month:
            picked.append(d)
            counters[m] += 1
    return picked

In [178]:
def saveDatesAndIntervals(out_dir: Path, dates, intervals):
    """Guarda las fechas en 'dates.txt' y los intervalos en 'intervals.json' dentro del directorio especificado."""
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "dates.txt").write_text("\n".join(dates), encoding="utf-8")
    with (out_dir / "intervals.json").open("w", encoding="utf-8") as f:
        json.dump({"intervals": intervals}, f, indent=2, ensure_ascii=False)

In [179]:
def executeSection1():
    """Genera intervalos diarios a partir de las fechas configuradas y guarda los resultados en archivos de salida."""

    if USE_SUBSET:
        dates = pickDatesPerMonth(PROF_DATES, SUBSET_MONTHS, PER_MONTH)
    else:
        dates = PROF_DATES

    intervals = buildDailyIntervals(dates)
    saveDatesAndIntervals(DATA_DIR, dates, intervals)

    print(f"[OK] Fechas: {len(dates)} -> guardadas en {DATA_DIR/'dates.txt'}")
    print(f"[OK] Intervalos: {len(intervals)} -> guardados en {DATA_DIR/'intervals.json'}")
    print("Ejemplo primer intervalo:", intervals[0] if intervals else "N/A")


In [180]:
if DO_PIPELINE:
    executeSection1()

[OK] Fechas: 29 -> guardadas en d:\repositorios\UVG\2025\labs-ds\lab4\data\dates.txt
[OK] Intervalos: 29 -> guardados en d:\repositorios\UVG\2025\labs-ds\lab4\data\intervals.json
Ejemplo primer intervalo: ['2025-02-07T00:00:00Z', '2025-02-08T00:00:00Z']


## Conexión, AOI y bandas (incisos 1, 2 y parte de 3)

**Objetivo:**

* Crear conexión a **openEO/CDSE**.
* Cargar el **GeoJSON** de cada lago y obtener su **geometría** (no bbox) para recortar en el servidor.
* Cargar **SENTINEL2\_L2A** con filtro de nubosidad (*server-side*).
* Pedir solo las **bandas necesarias** según el índice:

  * **RGB:** B04, B03, B02 (vista real).
  * **NDVI/NDWI:** B08 (NIR), B04 (Red), B03 (Green).
  * **Cianobacteria (NDCI + máscara agua):** B02, B03, B04, B05, B08, B8A, B11, B12.

**Por qué:**

* Recortar en el servidor evita descargar zonas irrelevantes.
* Menos bandas = menos datos y mayor velocidad; pero para el índice de cianobacteria se necesitan todas las que indica el script.

**Resultado:**

* Obtener un *DataCube* por lago, con geometría aplicada y solo las bandas/fechas necesarias, ya **recortado** y opcionalmente **re-muestreado** (p. ej. 20 m) para acelerar el procesamiento.

In [181]:
def connectToSentinelHub():
    """Conecta con el endpoint de openEO/CDSE y autentica mediante OIDC."""
    return openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()


In [182]:
def readLakeGeometry(geojson_path: str):
    """Lee un GeoJSON y devuelve la geometría unificada en formato dict (EPSG:4326)."""
    gdf = gpd.read_file(geojson_path)
    if gdf.crs is None:
        gdf.set_crs(4326, inplace=True)
    else:
        gdf = gdf.to_crs(4326)

    geom = gdf.geometry.union_all()

    return mapping(geom)   # dict con {"type": "...", "coordinates": ...}


In [183]:
def loadS2BaseCube(conn, geometry, start_date, end_date, bands):
    """Carga un DataCube Sentinel-2 L2A filtrado por fecha, AOI, bandas y nubosidad, 
    lo escala a reflectancia y lo remuestrea a la resolución indicada."""
    cube = conn.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=geometry,
        temporal_extent=[start_date, end_date],
        bands=bands,
        max_cloud_cover=MAX_CLOUD
    )
    cube = cube * 0.0001
    cube = cube.resample_spatial(resolution=RESOLUTION)
    return cube


In [184]:
def aggregateByIntervals(cube, intervals):
    """Agrega el DataCube en intervalos temporales dados usando la mediana como reductor."""
    return cube.aggregate_temporal(intervals=intervals, reducer="median") # intervals = [[startZ, endZ], ...]


In [185]:
def buildNdviCube(cube):
    """Calcula el índice NDVI (vegetación) a partir del DataCube Sentinel-2."""
    nir = cube.band("B08"); red = cube.band("B04")
    return (nir - red) / (nir + red)

def buildNdwiCube(cube):
    """Calcula el índice NDWI (agua) a partir del DataCube Sentinel-2."""
    green = cube.band("B03"); nir = cube.band("B08")
    return (green - nir) / (green + nir)

def buildCyanoChlaCube(cube):
    """Calcula la concentración de clorofila-a cianobacteriana usando múltiples índices y máscaras de agua."""
    blue  = cube.band("B02"); green = cube.band("B03"); red = cube.band("B04")
    b05   = cube.band("B05"); nir   = cube.band("B08"); b8a  = cube.band("B8A")
    swir1 = cube.band("B11"); swir2 = cube.band("B12")

    ndvi = (nir - red) / (nir + red)
    mndwi = (green - swir1) / (green + swir1)
    ndwi  = (green - nir)   / (green + nir)
    ndwi_leaves = (nir - swir1) / (nir + swir1)
    aweish  = blue + 2.5*green - 1.5*(nir + swir1) - 0.25*swir2
    aweinsh = 4*(green - swir1) - (0.25*nir + 2.75*swir1)
    dbsi = ((swir1 - green) / (swir1 + green)) - ndvi

    water = (mndwi > 0.42) | (ndwi > 0.4) | (aweinsh > 0.1879) | (aweish > 0.1112) | (ndvi < -0.2) | (ndwi_leaves > 1)
    water = water & ~((aweinsh <= -0.03) | (dbsi > 0))

    ndci = (b05 - red) / (b05 + red)
    chl  = 826.57 * (ndci ** 3) - 176.43 * (ndci ** 2) + 19 * ndci + 4.071
    return chl * water

In [186]:
def downloadCubeAsTiff(conn, cube, out_dir: Path):
    """Descarga un DataCube como archivos GeoTIFF en el directorio especificado."""
    out_dir.mkdir(parents=True, exist_ok=True)
    result = cube.save_result(format="GTIFF")
    job = conn.create_job(result)
    job.start_and_wait()
    job.get_results().download_files(str(out_dir))


In [ ]:
def executeSection2():
    """Carga, procesa y descarga índices satelitales para los lagos Atitlán y Amatitlán."""

    # Fechas/intervalos del p2.py
    intervals_path = DATA_DIR / "intervals.json"
    dates_path = DATA_DIR / "dates.txt"
    assert intervals_path.exists(), "Falta data/intervals.json (ejecuta p2.py)"
    with intervals_path.open("r", encoding="utf-8") as f:
        INTERVALS = json.load(f)["intervals"]

    # AOI
    atitlan_geom   = readLakeGeometry(str(DATA_DIR / "Lago_Atitlan.geojson"))
    amatitlan_geom = readLakeGeometry(str(DATA_DIR / "Lago_Amatitlan.geojson"))

    # Rango amplio (cubrir todas las fechas del archivo)
    start_date = INTERVALS[0][0][:10]
    end_date   = INTERVALS[-1][1][:10]

    conn = connectToSentinelHub()

    # --- Cargas por lago para índices ---
    atitlan_base   = loadS2BaseCube(conn, atitlan_geom,   start_date, end_date, BANDS_INDEX)
    amatitlan_base = loadS2BaseCube(conn, amatitlan_geom, start_date, end_date, BANDS_INDEX)

    # Agregación por las fechas exactas (mosaico diario)
    atitlan_daily   = aggregateByIntervals(atitlan_base,   INTERVALS)
    amatitlan_daily = aggregateByIntervals(amatitlan_base, INTERVALS)

    # Índices en la nube
    cy_atitlan     = buildCyanoChlaCube(atitlan_daily)
    cy_amatitlan   = buildCyanoChlaCube(amatitlan_daily)
    ndvi_atitlan   = buildNdviCube(atitlan_daily)
    ndvi_amatitlan = buildNdviCube(amatitlan_daily)
    ndwi_atitlan   = buildNdwiCube(atitlan_daily)
    ndwi_amatitlan = buildNdwiCube(amatitlan_daily)

    # Descargas
    downloadCubeAsTiff(conn, cy_atitlan,      DATA_DIR / "Cyano_Atitlan")
    downloadCubeAsTiff(conn, cy_amatitlan,    DATA_DIR / "Cyano_Amatitlan")
    downloadCubeAsTiff(conn, ndvi_atitlan,    DATA_DIR / "NDVI_Atitlan")
    downloadCubeAsTiff(conn, ndvi_amatitlan,  DATA_DIR / "NDVI_Amatitlan")
    downloadCubeAsTiff(conn, ndwi_atitlan,    DATA_DIR / "NDWI_Atitlan")
    downloadCubeAsTiff(conn, ndwi_amatitlan,  DATA_DIR / "NDWI_Amatitlan")

    # (Opcional) preview RGB
    if DO_RGB_PREVIEW:
        atitlan_rgb_base   = loadS2BaseCube(conn, atitlan_geom,   start_date, end_date, BANDS_RGB)
        amatitlan_rgb_base = loadS2BaseCube(conn, amatitlan_geom, start_date, end_date, BANDS_RGB)
        atitlan_rgb_daily   = aggregateByIntervals(atitlan_rgb_base,   INTERVALS)
        amatitlan_rgb_daily = aggregateByIntervals(amatitlan_rgb_base, INTERVALS)
        downloadCubeAsTiff(conn, atitlan_rgb_daily,   DATA_DIR / "RGB_Atitlan")
        downloadCubeAsTiff(conn, amatitlan_rgb_daily, DATA_DIR / "RGB_Amatitlan")

    print("[OK] s2 terminado. GeoTIFFs en data/*")

In [188]:
if DO_PIPELINE:
    executeSection2()

Authenticated using refresh token.
0:00:00 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': send 'start'
0:00:13 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': created (progress 0%)
0:00:18 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': created (progress 0%)
0:00:24 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': running (progress N/A)
0:00:33 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': running (progress N/A)
0:00:43 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': running (progress N/A)
0:00:55 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': running (progress N/A)
0:01:11 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': running (progress N/A)
0:01:30 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': running (progress N/A)
0:01:54 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': running (progress N/A)
0:02:24 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': running (progress N/A)
0:03:02 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': running (progress N/A)
0:03:48 Job 'j-2508100409504fb9b29bf0e8fafcbc8e': running (progress N/A)
0:04:47 Job 'j-2508100409504

## Aplicar el script de cianobacteria en la nube (Sentinel Hub Process API)

**Objetivo:**

* Leer la **AOI (Área de Interés)** desde archivos **GeoJSON** ubicados en `data`.
* Cargar los intervalos de fechas desde `intervals.json` generado por `p2.py`.
* Ejecutar un **evalscript** que calcula el índice numérico "Cyanobacteria Chlorophyll-a NDCI" para detectar clorofila.
* Descargar un archivo **TIFF** por cada fecha e intervalo para cada lago, con la clorofila en una sola banda, en formato **FLOAT32**.

**Por qué:**

* Automatizar la descarga de productos derivados de Sentinel-2 para monitoreo de cianobacterias.
* Usar la API de Sentinel Hub permite procesamiento en la nube, evitando descarga y manejo de imágenes brutas grandes.
* Descargar sólo las fechas e intervalos relevantes, reduciendo almacenamiento y tiempos.

**Resultado:**

* Archivos TIFF con mapas de clorofila, organizados por lago y fecha en la carpeta `data/<Lago>/Cyano_SH/`.
* Cada TIFF contiene una banda con valores de clorofila calculados a partir del índice NDCI y una máscara de agua, listos para análisis posteriores.

In [189]:
def getConfig():
    """
    Obtiene la configuración de SHConfig con valores de variables de entorno.
    Valida que CLIENT_ID y CLIENT_SECRET estén definidos.
    """
    cfg = SHConfig()
    cfg.sh_client_id     = os.getenv("CLIENT_ID", cfg.sh_client_id)
    cfg.sh_client_secret = os.getenv("CLIENT_SECRET", cfg.sh_client_secret)
    if os.getenv("SH_TOKEN_URL"):
        cfg.sh_token_url = os.environ["SH_TOKEN_URL"]
    assert cfg.sh_client_id and cfg.sh_client_secret, "Configura CLIENT_ID/CLIENT_SECRET en .env"
    return cfg

In [190]:
def readLakeGeometry(path_geojson: Path):
    """
    Lee un archivo GeoJSON y devuelve la geometría combinada en CRS 4326.
    """
    gdf = gpd.read_file(path_geojson)
    gdf = gdf.set_crs(4326) if gdf.crs is None else gdf.to_crs(4326)
    return gdf.geometry.union_all()

In [191]:
def requestForInterval(geom_shp, time_start, time_end, evalscript, cfg, out_dir: Path, target_filename: str):
    """
    Realiza una solicitud a Sentinel Hub para un intervalo de tiempo y guarda el resultado TIFF.
    """
    # BBox WGS84
    minx, miny, maxx, maxy = geom_shp.bounds
    bbox_wgs84 = BBox((minx, miny, maxx, maxy), crs=CRS.WGS84)
    # Dimensiones a ~RESOLUTION m/px (calcular en 3857)
    bbox_3857 = bbox_wgs84.transform(CRS.POP_WEB)
    width, height = bbox_to_dimensions(bbox_3857, RESOLUTION)

    req = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,
                time_interval=(time_start, time_end),
                other_args={"dataFilter": {"maxCloudCoverage": MAX_CLOUD}}
            )
        ],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=bbox_wgs84,
        size=(width, height),
        config=cfg,
    )

    # Descarga con reintentos/backoff ante rate limit
    attempt = 0
    success = False
    data = None

    while not success and attempt <= MAX_RETRIES:
        try:
            data = req.get_data(save_data=False)
            success = True
        except DownloadFailedException as e:
            msg = str(e).lower()
            if attempt < MAX_RETRIES and ("rate limit" in msg or "429" in msg or "too many requests" in msg):
                wait = BASE_BACKOFF_SEC * (2 ** attempt)
                time.sleep(wait)
                attempt += 1
            else:
                raise

    if not success or not data or data[0] is None:
        raise RuntimeError(f"Sin datos para {time_start}–{time_end}")

    arr = data[0]
    if arr.ndim == 3 and arr.shape[2] == 1:
        arr = arr[:, :, 0]
    arr = arr.astype(np.float32)

    out_dir.mkdir(parents=True, exist_ok=True)
    dest = out_dir / target_filename
    transform = from_bounds(minx, miny, maxx, maxy, width, height)
    profile = {
        "driver": "GTiff",
        "height": arr.shape[0],
        "width": arr.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": RIO_CRS.from_epsg(4326),
        "transform": transform,
        "compress": "deflate",
        "predictor": 2,
        "tiled": True,
        "nodata": np.nan,
    }

    with rasterio.open(dest, "w", **profile) as dst:
        dst.write(arr, 1)

    return dest

In [192]:
def processLake(name: str, geojson_path: Path, intervals, out_root: Path, cfg):
    """
    Procesa un lago descargando datos para intervalos de tiempo y guardándolos en archivos TIFF.
    """
    assert geojson_path.exists(), f"No existe: {geojson_path}"
    geom = readLakeGeometry(geojson_path)
    lake_out = out_root / name / OUT_SUBDIR
    lake_out.mkdir(parents=True, exist_ok=True)

    for i, (startZ, endZ) in enumerate(intervals, 1):
        date_label = startZ[:10]
        print(f"[{name}] {i}/{len(intervals)} -> {date_label}")
        requestForInterval(geom, startZ, endZ, EVALSCRIPT_CHL, cfg, lake_out, f"{name}_chl_{date_label}.tif")
        time.sleep(PAUSE_BETWEEN_REQUESTS)  # pausa fija entre requests

In [ ]:
def executeSection3():
    """Descarga datos de clorofila para lagos específicos en intervalos definidos y guarda los archivos TIFF."""
    intervals = json.loads((DATA_DIR / "intervals.json").read_text(encoding="utf-8"))["intervals"]
    cfg = getConfig()

    processLake("Atitlan",   DATA_DIR / "Lago_Atitlan.geojson",   intervals, DATA_DIR, cfg)
    processLake("Amatitlan", DATA_DIR / "Lago_Amatitlan.geojson", intervals, DATA_DIR, cfg)

    print("[OK] s3 completado. TIFFs en data/<Lago>/Cyano_SH/*.tif")

In [196]:
if DO_PIPELINE:
    executeSection3()

[Atitlan] 1/29 -> 2025-02-07
[Atitlan] 2/29 -> 2025-02-10
[Atitlan] 3/29 -> 2025-02-25
[Atitlan] 4/29 -> 2025-02-27
[Atitlan] 5/29 -> 2025-03-02
[Atitlan] 6/29 -> 2025-03-04
[Atitlan] 7/29 -> 2025-03-07
[Atitlan] 8/29 -> 2025-03-09
[Atitlan] 9/29 -> 2025-03-12
[Atitlan] 10/29 -> 2025-03-14
[Atitlan] 11/29 -> 2025-03-19
[Atitlan] 12/29 -> 2025-03-22
[Atitlan] 13/29 -> 2025-03-24
[Atitlan] 14/29 -> 2025-03-26
[Atitlan] 15/29 -> 2025-04-03
[Atitlan] 16/29 -> 2025-04-11
[Atitlan] 17/29 -> 2025-04-13
[Atitlan] 18/29 -> 2025-04-15
[Atitlan] 19/29 -> 2025-04-16
[Atitlan] 20/29 -> 2025-04-18
[Atitlan] 21/29 -> 2025-04-28
[Atitlan] 22/29 -> 2025-05-03
[Atitlan] 23/29 -> 2025-05-13
[Atitlan] 24/29 -> 2025-05-28
[Atitlan] 25/29 -> 2025-07-10
[Atitlan] 26/29 -> 2025-07-17
[Atitlan] 27/29 -> 2025-07-20
[Atitlan] 28/29 -> 2025-07-24
[Atitlan] 29/29 -> 2025-08-01
[Amatitlan] 1/29 -> 2025-02-07
[Amatitlan] 2/29 -> 2025-02-10
[Amatitlan] 3/29 -> 2025-02-25
[Amatitlan] 4/29 -> 2025-02-27
[Amatitlan] 5/2

## Exportación

**Objetivo:**

* Convertir los archivos GeoTIFF individuales descargados en GeoTIFFs multitemporales apilados (1 banda = 1 fecha).
* Inventariar las bandas con fechas para facilitar análisis posteriores.
* Generar archivos livianos y organizados para cada índice y lago.

**Por qué:**

* Reduce la cantidad de archivos al apilar fechas en un solo archivo.
* Facilita la gestión y análisis de los datos multitemporales.
* Facilita la revisión rápida con inventarios CSV.

**Resultado:**

* Carpetas `export_consolidated/` con GeoTIFFs multitemporales para NDVI, NDWI y CYANO por lago.
* Archivos CSV con inventarios de bandas y fechas para cada archivo TIFF multitemporal.

In [197]:
def readIntervals():
    """Lee y devuelve la lista de intervalos de fechas desde el archivo intervals.json."""
    with INTERVALS_PATH.open("r", encoding="utf-8") as f:
        return json.load(f)["intervals"]  # [[startZ,endZ], ...]

In [198]:
def listTiffs(folder: Path):
    """Lista y ordena todos los archivos .tif dentro del directorio dado."""
    return sorted([p for p in folder.glob("*.tif")])

In [199]:
def isSingleMultitemporal(tif_path: Path) -> bool:
    """Determina si el path es un único GeoTIFF multibanda (multitemporal)."""
    tiffs = listTiffs(tif_path) if tif_path.is_dir() else [tif_path]
    if len(tiffs) != 1:
        return False
    with rasterio.open(tiffs[0]) as src:
        return src.count > 1  # más de una banda

In [200]:
def stackManySinglesToMultitemporal(folder: Path, out_path: Path, dates: list):
    """
    Apila muchos TIFF (uno por fecha, 1 banda) -> 1 multibanda,
    REPROYECTANDO cada uno a la grilla del primero.
    """
    tiffs = listTiffs(folder)
    if not tiffs:
        raise RuntimeError(f"No .tif files in {folder}")

    # Orden por fecha en nombre
    def dateKey(p: Path):
        s = p.stem
        for d in dates:
            if d in s:
                return d
        return "9999-12-31"
    tiffs_sorted = sorted(tiffs, key=dateKey)

    # Grilla de referencia = primer TIFF
    with rasterio.open(tiffs_sorted[0]) as ref:
        ref_crs = ref.crs
        ref_transform = ref.transform
        ref_h, ref_w = ref.height, ref.width
        dtype = ref.dtypes[0]
        profile = ref.profile.copy()
        profile.update(count=len(tiffs_sorted), compress="deflate", predictor=2, tiled=True, nodata=np.nan)

    arrays = []
    for tif in tiffs_sorted:
        with rasterio.open(tif) as src:
            if (src.crs == ref_crs and src.transform == ref_transform
                and src.width == ref_w and src.height == ref_h):
                band = src.read(1).astype(np.float32)
            else:
                band = np.full((ref_h, ref_w), np.nan, dtype=np.float32)
                reproject(
                    source=src.read(1),
                    destination=band,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear,
                    dst_nodata=np.nan
                )
        arrays.append(band[np.newaxis, ...])

    stack = np.concatenate(arrays, axis=0)

    with rasterio.open(out_path, "w", **profile) as dst:
        for i in range(stack.shape[0]):
            dst.write(stack[i], i + 1)

    return [t.name for t in tiffs_sorted]

In [201]:
def buildBandDateInventory(dates: list, written_order_names=None, single_multitemporal_path: Path = None):
    """
    Genera pares (band_index, date, src_name).
    - Si se apiló desde muchos TIFF: usa written_order_names
    - Si ya era multitemporal: usa 'dates' en orden
    """
    rows = []
    if written_order_names:
        for i, name in enumerate(written_order_names, start=1):
            date = next((d for d in dates if d in name), "")
            rows.append((i, date, name))
    elif single_multitemporal_path:
        with rasterio.open(single_multitemporal_path) as src:
            n = src.count
        for i in range(1, n + 1):
            date = dates[i - 1] if i - 1 < len(dates) else ""
            rows.append((i, date, single_multitemporal_path.name))
    return rows

In [202]:
def processOne(lake: str, index_name: str, folder: Path, dates: list):
    """
    folder puede contener:
      - muchos TIFF de 1 banda (por fecha) -> se apilan a 1 multitemporal
      - 1 TIFF multibanda -> solo inventario
    """
    out_tif = OUT_DIR / f"{lake}_{index_name}_stack.tif"
    inv_csv = OUT_DIR / f"{lake}_{index_name}_bands_dates.csv"

    if not folder.exists():
        print(f"[WARN] No existe {folder} -> saltando {lake}-{index_name}")
        return

    if isSingleMultitemporal(folder):
        single = listTiffs(folder)[0]
        print(f"[INFO] {lake}-{index_name}: ya multitemporal -> {single.name}")
        if single.resolve() != out_tif.resolve():
            out_tif.write_bytes(single.read_bytes())
        rows = buildBandDateInventory(dates, single_multitemporal_path=single)
    else:
        print(f"[INFO] {lake}-{index_name}: apilando {len(listTiffs(folder))} TIFF -> {out_tif.name}")
        written_order_names = stackManySinglesToMultitemporal(folder, out_tif, dates)
        rows = buildBandDateInventory(dates, written_order_names=written_order_names)

    with inv_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["band_index", "date(YYYY-MM-DD)", "source"])
        for row in rows:
            w.writerow(row)

    print(f"[OK] {lake}-{index_name}: {out_tif.name}, inventario -> {inv_csv.name}")

In [203]:
def executeSection4():
    # fechas limpias (YYYY-MM-DD) a partir de intervals.json
    intervals = readIntervals()
    dates = [iv[0][:10] for iv in intervals]
    for (lake, idx), folder in DIRS.items():
        processOne(lake, idx, folder, dates)
    print(f"[DONE] Exportación consolidada en {OUT_DIR}/")

In [204]:
if DO_PIPELINE:
    executeSection4()

[INFO] Atitlan-NDVI: apilando 24 TIFF -> Atitlan_NDVI_stack.tif
[OK] Atitlan-NDVI: Atitlan_NDVI_stack.tif, inventario -> Atitlan_NDVI_bands_dates.csv
[INFO] Amatitlan-NDVI: apilando 18 TIFF -> Amatitlan_NDVI_stack.tif
[OK] Amatitlan-NDVI: Amatitlan_NDVI_stack.tif, inventario -> Amatitlan_NDVI_bands_dates.csv
[INFO] Atitlan-NDWI: apilando 24 TIFF -> Atitlan_NDWI_stack.tif
[OK] Atitlan-NDWI: Atitlan_NDWI_stack.tif, inventario -> Atitlan_NDWI_bands_dates.csv
[INFO] Amatitlan-NDWI: apilando 18 TIFF -> Amatitlan_NDWI_stack.tif
[OK] Amatitlan-NDWI: Amatitlan_NDWI_stack.tif, inventario -> Amatitlan_NDWI_bands_dates.csv
[INFO] Atitlan-CYANO: apilando 29 TIFF -> Atitlan_CYANO_stack.tif
[OK] Atitlan-CYANO: Atitlan_CYANO_stack.tif, inventario -> Atitlan_CYANO_bands_dates.csv
[INFO] Amatitlan-CYANO: apilando 29 TIFF -> Amatitlan_CYANO_stack.tif
[OK] Amatitlan-CYANO: Amatitlan_CYANO_stack.tif, inventario -> Amatitlan_CYANO_bands_dates.csv
[DONE] Exportación consolidada en d:\repositorios\UVG\2025\l

## Análisis temporal

**Objetivo:**

* Calcular métricas temporales robustas de índices espectrales (CYANO, NDVI, NDWI) dentro del polígono de cada lago.
* Para cada fecha, extraer el valor promedio o una métrica específica (percentil 90, área umbral) dentro del lago enmascarado.
* Guardar series temporales de valores medios por lago e índice.
* Detectar picos significativos en las series de CYANO para identificar eventos de floración.
* Generar gráficos temporales para visualización clara de la evolución.

**Por qué:**

* Permite monitorear la evolución y dinámica temporal de floraciones y otros fenómenos acuáticos.
* Facilita la identificación de fechas críticas con altos niveles de floración mediante detección de picos.
* Proporciona datos consolidados para análisis posteriores y toma de decisiones ambientales.

**Resultado:**

* Archivos CSV con series temporales limpias de índices para cada lago.
* Archivos JSON con fechas y valores de picos detectados en CYANO.
* Gráficos en PNG que muestran la evolución temporal de los índices por lago.
* Reportes impresos indicando el estado y los picos detectados.

In [205]:
def readBandDateInventory(csv_path: Path):
    """
    Lee un inventario CSV de bandas y devuelve listas con el índice de banda, la fecha y la fuente.
    """

    bands, dates, names = [], [], []
    with csv_path.open("r", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            bands.append(int(row["band_index"]))
            dates.append(row["date(YYYY-MM-DD)"])
            names.append(row["source"])
    return bands, dates, names

In [206]:
def computeMeanByDate(stack_path: Path, inv_csv: Path, lake_geojson_path: Path):
    """
    Calcula una métrica por fecha recortando al polígono del lago.
    - Convierte NODATA a NaN.
    - En CYANO: CLIP a >= 0 (no descartar ceros).
    - Métrica elegible por CYANO_METRIC: "area" | "p90" | "mean".
      * area: fracción de píxeles > CYANO_THRESH (aprox. área con bloom)
      * p90 : percentil 90 (intensidad alta)
      * mean: promedio sobre válidos
    - Exige cobertura mínima (>=1%) de píxeles válidos; si no, NaN.
    """
    bands, dates, _ = readBandDateInventory(inv_csv)
    lake_gdf = gpd.read_file(lake_geojson_path)

    vals_out = []
    with rasterio.open(stack_path) as src:
        # reproyecta AOI al CRS del raster
        if lake_gdf.crs is None:
            lake_gdf.set_crs(4326, inplace=True)
        lake_gdf = lake_gdf.to_crs(src.crs)
        geoms = [g.__geo_interface__ for g in lake_gdf.geometry]

        is_cyano_stack = "CYANO" in stack_path.name.upper()
        min_valid_frac = 0.01  # 1% de cobertura mínima

        for b in bands:
            try:
                clipped, _ = rio_mask(src, geoms, crop=True, filled=True,
                                      indexes=b, nodata=np.nan)
                a = clipped[0].astype(np.float32)

                # nodata explícito -> NaN
                nod = src.nodata
                if nod is not None and not (isinstance(nod, float) and math.isnan(nod)):
                    a = np.where(a == nod, np.nan, a)

                # no finitos -> NaN
                a[~np.isfinite(a)] = np.nan

                # CYANO: CLIP negativos a 0 (no los descartes)
                if is_cyano_stack:
                    a = np.clip(a, 0, None)

                valid_mask = np.isfinite(a)
                if valid_mask.mean() < min_valid_frac:
                    m = np.nan
                else:
                    valid_vals = a[valid_mask]
                    if is_cyano_stack and CYANO_METRIC.lower() == "area":
                        # fracción (0..1) de lago con valores > umbral
                        m = float((valid_vals > CYANO_THRESH).mean())
                    elif is_cyano_stack and CYANO_METRIC.lower() == "p90":
                        m = float(np.percentile(valid_vals, 90))
                    else:
                        m = float(valid_vals.mean())

                vals_out.append(m if np.isfinite(m) else np.nan)

            except Exception:
                vals_out.append(np.nan)

    return dates, vals_out

In [207]:
def detectPeaks(dates, values, method="percentile", p=90, zthr=1.0):
    """
    Detección de picos robusta:
    - Solo usa valores finitos.
    - Si hay <3 puntos válidos, no reporta picos.
    - Percentil: marca v >= percentil p de los válidos.
    - Z-score: marca z >= zthr sobre válidos.
    """
    arr = np.array(values, dtype=np.float32)
    finite = np.isfinite(arr)

    if finite.sum() < 3:
        return [], []

    if method == "percentile":
        thr = float(np.percentile(arr[finite], p))
        idx = [i for i, v in enumerate(arr) if np.isfinite(v) and v >= thr]
    else:
        mu = float(np.mean(arr[finite]))
        sd = float(np.std(arr[finite]))
        if not np.isfinite(sd) or sd == 0.0:
            return [], []
        z = (arr - mu) / sd
        idx = [i for i, zz in enumerate(z) if np.isfinite(zz) and zz >= zthr]

    return [dates[i] for i in idx], [float(arr[i]) for i in idx]

In [208]:
def saveSeriesCSV(lake, index_name, dates, values):
    """
    Guarda en un archivo CSV la serie temporal del índice promedio por fecha para un lago.
    """
    
    out_csv = OUT_DIR2 / f"{lake}_{index_name}_series.csv"
    with out_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["date", f"{index_name}_mean"])
        for d, v in zip(dates, values):
            w.writerow([d, v if (v is not None and np.isfinite(v)) else ""])
    return out_csv

In [209]:
def plotTimeSeries(lake, series_dict):    
    """
    Genera y guarda un gráfico de líneas que muestra la evolución temporal de índices promedio para un lago.
    """
    
    plt.figure(figsize=(10, 5))
    for idx_name, (dates, vals) in series_dict.items():
        if not dates: 
            continue
        x = np.arange(len(dates)); y = np.array(vals, dtype=np.float32)
        plt.plot(x, y, label=idx_name)
    if series_dict:
        any_dates = next(iter(series_dict.values()))[0]
        if any_dates:
            xticks = np.arange(len(any_dates))
            plt.xticks(xticks, any_dates, rotation=45, ha="right")
    plt.title(f"{lake} - Índices vs tiempo"); plt.xlabel("Fecha"); plt.ylabel("Valor medio en lago")
    plt.legend(); plt.tight_layout()
    fig_path = OUT_DIR2 / f"{lake}_time_series.png"
    plt.savefig(fig_path, dpi=150); plt.close()
    return fig_path

In [210]:
def executeSection5():
    """
    Ejecuta el análisis temporal (Sección 5), calculando series de índices (CYANO, NDVI, NDWI) por lago, guardando CSV y gráficas, y detectando picos de proliferación.
    """
    
    intervals_path = DATA_DIR / "intervals.json"
    dates_master = None
    if intervals_path.exists():
        with intervals_path.open("r", encoding="utf-8") as f:
            intervals = json.load(f)["intervals"]
        dates_master = [iv[0][:10] for iv in intervals]

    for lake, geojson in LAKES.items():
        series = {}

        for idx_name in ("CYANO","NDVI","NDWI"):
            key = (lake, idx_name)
            if key not in STACKS: 
                continue
            tif, inv = STACKS[key]
            if not (tif.exists() and inv.exists()):
                continue

            d, v = computeMeanByDate(tif, inv, geojson)
            if dates_master and idx_name == "CYANO":
                order = {d_: i for i, d_ in enumerate(dates_master)}
                pairs = [(dd, vv) for dd, vv in zip(d, v) if dd in order]
                pairs.sort(key=lambda x: order[x[0]])
                d = [p[0] for p in pairs]; v = [p[1] for p in pairs]
            series[idx_name] = (d, v)
            saveSeriesCSV(lake, idx_name, d, v)

            if idx_name == "CYANO":
                p_dates, p_vals = detectPeaks(d, v, method=PEAK_METHOD, p=PEAK_PERCENTILE, zthr=ZSCORE_THRESHOLD)
                (OUT_DIR2 / f"{lake}_CYANO_peaks.json").write_text(
                    json.dumps({"peak_dates": p_dates, "peak_values": p_vals}, indent=2, ensure_ascii=False),
                    encoding="utf-8"
                )

        if series:
            fig_path = plotTimeSeries(lake, series)
            print(f"[OK] {lake}: series y figura -> {fig_path}")

        if "CYANO" in series:
            rep = json.loads((OUT_DIR2 / f"{lake}_CYANO_peaks.json").read_text(encoding="utf-8"))
            if rep["peak_dates"]:
                print(f"[PEAKS] {lake} Cyano:", list(zip(rep["peak_dates"], np.round(rep["peak_values"], 3).tolist())))
            else:
                print(f"[PEAKS] {lake} Cyano: sin picos según criterio {PEAK_METHOD}")

    print("[DONE] s5: series, picos y gráficas generadas en data/analysis/")

In [211]:
if DO_PIPELINE:
    executeSection5()

[OK] Atitlan: series y figura -> d:\repositorios\UVG\2025\labs-ds\lab4\data\analysis\Atitlan_time_series.png
[PEAKS] Atitlan Cyano: [('2025-03-07', 145.173), ('2025-07-20', 142.293), ('2025-08-01', 170.228)]
[OK] Amatitlan: series y figura -> d:\repositorios\UVG\2025\labs-ds\lab4\data\analysis\Amatitlan_time_series.png
[PEAKS] Amatitlan Cyano: [('2025-02-07', 43.655), ('2025-08-01', 167.333)]
[DONE] s5: series, picos y gráficas generadas en data/analysis/


## Análisis espacial

**Objetivo:**

* Generar mapas que muestran la distribución espacial de la clorofila (índice Cyano) en lagos específicos.
* Crear tres tipos de visualizaciones:

  * **Mapas estáticos** usando Matplotlib para fechas seleccionadas, mostrando la concentración de clorofila en cada lago.
  * **Mapas interactivos** con Folium, donde se superpone el raster del índice Cyano sobre el contorno geográfico del lago para exploración dinámica.
  * **Comparativos 2×2**, que muestran una cuadrícula con cuatro fechas del mismo lago para visualizar cambios temporales.

**Por qué:**

* Observar la **distribución espacial** y patrones de floraciones de cianobacterias dentro del lago.
* Facilitar la comparación visual entre diferentes fechas para evaluar evolución y eventos relevantes.
* Usar mapas interactivos para una exploración más intuitiva y detallada del área afectada.

**Entrada:**

* Archivos raster con stacks de bandas de índice Cyano (TIFF multibanda).
* Inventarios CSV con fechas y bandas correspondientes.
* GeoJSON con el polígono de cada lago para recorte espacial.

**Salida:**

* Imágenes PNG con mapas estáticos de concentración de clorofila para fechas específicas.
* Imágenes PNG con grillas 2×2 para comparativos visuales.
* Archivos HTML con mapas interactivos Folium, incluyendo overlay del índice y contorno del lago.
* Todos los resultados se guardan en `data/maps/`.

**Resultado:**

* Visualizaciones claras que muestran patrones espaciales y temporales de la clorofila en los lagos Atitlán y Amatitlán.
* Herramientas para análisis exploratorio local, complementando análisis previos en la nube.

In [212]:
def readInventory(csv_path: Path):
    """
    Lee un inventario CSV y devuelve listas con los índices de banda y sus fechas correspondientes.
    """

    import csv
    bands, dates = [], []
    with csv_path.open("r", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            bands.append(int(row["band_index"]))
            dates.append(row["date(YYYY-MM-DD)"])
    return bands, dates

In [213]:
def getBandForDate(target_date: str, dates_list: list, bands_list: list) -> int:
    """
    Devuelve el índice de banda asociado a una fecha específica del inventario.
    """

    for b, d in zip(bands_list, dates_list):
        if d == target_date:
            return b
    raise ValueError(f"Fecha {target_date} no encontrada en inventario.")

In [214]:
def lake_geoms_in_raster_crs(geojson_path: Path, raster_crs):
    """
    Lee un GeoJSON de lagos y devuelve sus geometrías transformadas al sistema de referencia (CRS) del ráster.
    """

    gdf = gpd.read_file(geojson_path)
    if gdf.crs is None:
        gdf.set_crs(4326, inplace=True)
    if raster_crs is not None and gdf.crs != raster_crs:
        gdf = gdf.to_crs(raster_crs)
    geoms = [g.__geo_interface__ for g in gdf.geometry]
    return geoms, gdf


In [215]:
def clipBandToLake(src, band_index: int, lake_geoms: list):
    """
    Recorta una banda específica de un ráster a la geometría del lago, devolviendo el array recortado y su transformada.
    """
    clipped, out_transform = rio_mask(
        src, lake_geoms, crop=True, filled=True, indexes=[band_index], nodata=np.nan
    )
    a = clipped[0].astype(np.float32)  # (rows, cols)
    return a, out_transform

In [216]:
def robustMinMax(a: np.ndarray, low=2, high=98):
    """
    Calcula un rango robusto mínimo y máximo basado en percentiles (por defecto 2 y 98) ignorando valores no finitos,
    para evitar que valores extremos afecten el rango.
    """

    valid = a[np.isfinite(a)]
    if valid.size == 0:
        return 0.0, 1.0
    vmin, vmax = np.percentile(valid, [low, high])
    if vmin == vmax:
        vmax = vmin + 1e-6
    return float(vmin), float(vmax)

In [217]:
def boundsFromTransform(arr: np.ndarray, transform, crs):
    """
    Calcula los límites geográficos (bounds) en coordenadas WGS84 (latitud/longitud) a partir de un arreglo raster y su transformada espacial,
    convertiendo el sistema de referencia si es necesario para compatibilidad con visualización en Folium.
    """

    h, w = arr.shape
    minx, miny, maxx, maxy = array_bounds(h, w, transform)  # en CRS del raster
    if crs and getattr(crs, "to_epsg", lambda: None)() != 4326:
        minx, miny, maxx, maxy = transform_bounds(crs, "EPSG:4326", minx, miny, maxx, maxy, densify_pts=21)
    return (miny, minx, maxy, maxx)  # (south, west, north, east)

In [218]:
def _map_center_latlon(gdf):
    """
    Calcula el centro geográfico (latitud, longitud) del bounding box de un GeoDataFrame, asegurando que esté en el sistema de referencia WGS84.
    """

    g84 = gdf.to_crs(4326) if (gdf.crs and gdf.crs.to_epsg() != 4326) else gdf
    minx, miny, maxx, maxy = g84.total_bounds
    return [(miny + maxy) / 2.0, (minx + maxx) / 2.0]

In [219]:
def saveStaticMap(lake_name: str, date_str: str, stack_path: Path, inv_csv: Path, lake_geojson: Path):
    """
    Genera y guarda un mapa estático de clorofila-a (Cyano) para un lago en una fecha específica usando datos raster y geometrías de lago.
    """

    bands, dates = readInventory(inv_csv)

    with rasterio.open(stack_path) as src:
        lake_geoms, _ = lake_geoms_in_raster_crs(lake_geojson, src.crs)
        band = getBandForDate(date_str, dates, bands)
        arr, _ = clipBandToLake(src, band, lake_geoms)

    vmin, vmax = robustMinMax(arr)
    plt.figure(figsize=(7, 6))
    plt.imshow(arr, vmin=vmin, vmax=vmax)
    plt.title(f"{lake_name} – Cyano (chl) – {date_str}")
    plt.axis('off')
    cb = plt.colorbar()
    cb.set_label("Chlorophyll-a (aprox.)")
    out_png = OUT_DIR3 / f"{lake_name}_CYANO_{date_str}.png"
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()
    return out_png

In [220]:
def saveCompareGrid(lake_name: str, dates4: list, stack_path: Path, inv_csv: Path, lake_geojson: Path):
    """
    Genera y guarda una cuadrícula comparativa 2x2 de mapas estáticos de clorofila-a (Cyano) para un lago en cuatro fechas específicas.
    """

    assert len(dates4) == 4, "Debes pasar exactamente 4 fechas."
    bands, dates = readInventory(inv_csv)

    imgs = []
    with rasterio.open(stack_path) as src:
        lake_geoms, _ = lake_geoms_in_raster_crs(lake_geojson, src.crs)
        for d in dates4:
            b = getBandForDate(d, dates, bands)
            a, _ = clipBandToLake(src, b, lake_geoms)
            imgs.append(a)

    all_vals = np.concatenate([a.flatten() for a in imgs])
    vmin, vmax = robustMinMax(all_vals)

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes = axes.ravel()
    for ax, a, d in zip(axes, imgs, dates4):
        im = ax.imshow(a, vmin=vmin, vmax=vmax)
        ax.set_title(d)
        ax.axis('off')
    cbar = fig.colorbar(im, ax=axes.tolist(), fraction=0.02, pad=0.02)
    cbar.set_label("Chlorophyll-a (aprox.)")
    plt.suptitle(f"{lake_name} – Cyano comparativo 2×2", y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    out_png = OUT_DIR3 / f"{lake_name}_CYANO_compare_2x2.png"
    plt.savefig(out_png, dpi=150)
    plt.close()
    return out_png

In [221]:
def saveFoliumMap(lake_name: str, date_str: str, stack_path: Path, inv_csv: Path, lake_geojson: Path):
    """
    Genera y guarda un mapa interactivo de Folium con un overlay de clorofila-a (Cyano) para un lago en una fecha dada.
    """

    bands, dates = readInventory(inv_csv)

    with rasterio.open(stack_path) as src:
        lake_geoms, lake_gdf = lake_geoms_in_raster_crs(lake_geojson, src.crs)
        band = getBandForDate(date_str, dates, bands)
        arr, out_transform = clipBandToLake(src, band, lake_geoms)
        south, west, north, east = boundsFromTransform(arr, out_transform, src.crs)

    vmin, vmax = robustMinMax(arr)
    norm = (arr - vmin) / (vmax - vmin)
    norm = np.clip(norm, 0, 1)

    # PNG temporal para overlay
    fig = plt.figure(frameon=False)
    fig.set_size_inches(6, 5)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    ax.imshow(norm)
    tmp_png = OUT_DIR3 / f"tmp_{lake_name}_{date_str}.png"
    fig.savefig(tmp_png, dpi=150)
    plt.close(fig)

    # Centro del mapa (bbox) en WGS84
    center = _map_center_latlon(lake_gdf)
    m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

    folium.raster_layers.ImageOverlay(
        image=str(tmp_png),
        bounds=[[south, west], [north, east]],
        opacity=0.7,
        name=f"Cyano {date_str}"
    ).add_to(m)

    folium.GeoJson(lake_gdf.to_crs(4326).to_json(), name="Lago").add_to(m)

    cmap = bcm.LinearColormap(
        colors=['#440154', '#3B528B', '#21908C', '#5DC863', '#FDE725'],
        vmin=vmin, vmax=vmax
    )
    cmap.caption = f"Chlorophyll-a (aprox.) – {date_str}"
    cmap.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    out_html = OUT_DIR3 / f"{lake_name}_CYANO_{date_str}.html"
    m.save(str(out_html))

    try:
        tmp_png.unlink()
    except Exception:
        pass

    return out_html

In [222]:
def executeSection6():
    """
    Ejecuta la sección 6 del análisis, generando mapas estáticos, comparativos e interactivos para lagos.

    - Para cada lago en LAKES, verifica que existan los archivos necesarios (stack e inventario) para el índice CYANO.
    - Selecciona hasta 4 fechas preferidas de ejemplo o del inventario.
    - Genera un mapa estático para la primera fecha.
    - Si hay al menos 4 fechas, genera un mapa comparativo 2*2 con esas fechas.
    - Genera un mapa interactivo para la segunda fecha (o la primera si no hay segunda).
    """

    examples = {
        "Atitlan":   ["2025-04-03", "2025-05-03", "2025-07-10", "2025-08-01"],
        "Amatitlan": ["2025-04-03", "2025-05-28", "2025-07-17", "2025-08-01"],
    }

    for lake, geojson in LAKES.items():
        stack, inv = STACKS2[(lake, "CYANO")]
        if not (stack.exists() and inv.exists()):
            print(f"[WARN] Faltan archivos para {lake}-CYANO ({stack.name} / {inv.name})")
            continue

        # Fechas disponibles y selección
        _, available_dates = readInventory(inv)
        desired = examples.get(lake, [])
        chosen = [d for d in desired if d in available_dates]
        for d in available_dates:
            if len(chosen) >= 4:
                break
            if d not in chosen:
                chosen.append(d)

        if not chosen:
            print(f"[WARN] No hay fechas en inventario para {lake}.")
            continue

        first_date = chosen[0]
        png_path = saveStaticMap(lake, first_date, stack, inv, geojson)
        print(f"[OK] Estático: {png_path}")

        if len(chosen) >= 4:
            comp_png = saveCompareGrid(lake, chosen[:4], stack, inv, geojson)
            print(f"[OK] Comparativo 2x2: {comp_png}")
        else:
            print(f"[INFO] {lake}: menos de 4 fechas disponibles; se omite el comparativo.")

        second_date = chosen[1] if len(chosen) >= 2 else chosen[0]
        html_path = saveFoliumMap(lake, second_date, stack, inv, geojson)
        print(f"[OK] Interactivo: {html_path}")

    print("[DONE] s6: mapas generados en data/maps/")

In [223]:
if DO_PIPELINE:
    executeSection6()

[OK] Estático: d:\repositorios\UVG\2025\labs-ds\lab4\data\maps\Atitlan_CYANO_2025-04-03.png


C:\Users\josue\AppData\Local\Temp\ipykernel_45240\888459919.py:29: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.96])


[OK] Comparativo 2x2: d:\repositorios\UVG\2025\labs-ds\lab4\data\maps\Atitlan_CYANO_compare_2x2.png
[OK] Interactivo: d:\repositorios\UVG\2025\labs-ds\lab4\data\maps\Atitlan_CYANO_2025-05-03.html
[OK] Estático: d:\repositorios\UVG\2025\labs-ds\lab4\data\maps\Amatitlan_CYANO_2025-04-03.png


C:\Users\josue\AppData\Local\Temp\ipykernel_45240\888459919.py:29: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.96])


[OK] Comparativo 2x2: d:\repositorios\UVG\2025\labs-ds\lab4\data\maps\Amatitlan_CYANO_compare_2x2.png
[OK] Interactivo: d:\repositorios\UVG\2025\labs-ds\lab4\data\maps\Amatitlan_CYANO_2025-05-28.html
[DONE] s6: mapas generados en data/maps/


## Correlaciones Cyano – NDVI/NDWI

### ¿Qué hace este script?

Este script analiza la relación entre las concentraciones de cianobacterias (Cyano) y dos índices satelitales relacionados con la vegetación y el agua en dos lagos: Atitlán y Amatitlán.

### Pasos principales:

1. **Carga de datos:**

   * Se leen series temporales de promedios diarios para tres índices: CYANO, NDVI y NDWI.
   * Los datos provienen de archivos CSV, donde cada fila corresponde a un día y un valor promedio.

2. **Alineación por fechas:**

   * Dado que las fechas entre índices pueden no coincidir exactamente, se hace una intersección para que solo se comparen valores de los mismos días.
   * Esto permite comparar correctamente los valores de Cyano con NDVI y NDWI del mismo día.

3. **Cálculo de correlaciones:**

   * Se calculan dos tipos de correlación entre Cyano y cada índice:

     * **Pearson:** mide la correlación lineal entre las variables.
     * **Spearman:** mide la correlación en el orden o rangos de los datos (más robusta ante no linealidades).
   * Se reporta el coeficiente y el número de pares de datos usados.

4. **Visualización:**

   * Se generan gráficos de dispersión (scatter plots) de Cyano vs NDVI y Cyano vs NDWI para cada lago.
   * Estos gráficos ayudan a visualizar la relación y posible tendencia entre los índices.

5. **Comparación de picos de Cyano:**

   * Se cargan datos de fechas y valores máximos de picos detectados de Cyano.
   * Se resumen el número de picos y el valor máximo para cada lago en un JSON.

6. **Resultados finales:**

   * Se guardan los resultados de correlaciones en `correlations_summary.json`.
   * Se guarda un resumen de picos en `peaks_summary.json`.
   * Imprime mensajes indicando que el análisis terminó correctamente.

### Interpretación y contexto:

* **Relación Cyano vs NDWI:**

  * Un aumento en Cyano puede coincidir con un aumento en NDWI, lo que indica más agua superficial o mayor humedad. Esto tiene sentido porque las cianobacterias crecen en la superficie del agua.

* **Relación Cyano vs NDVI:**

  * La correlación puede ser menos clara o incluso negativa, porque NDVI representa la vegetación terrestre o ribereña, no la algal en el agua.
  * NDVI alto cerca de la orilla no necesariamente implica aumento de Cyano, ya que son fenómenos diferentes.

* **Conclusión:**

  * La clave está en cómo varían juntos estos índices, no solo en valores absolutos.
  * Las correlaciones ayudan a entender las dinámicas ecológicas en cada lago.

In [224]:
def loadSeries(lake, index_name):
    """
    Lee el archivo CSV ubicado en la ruta ANLYSIS/{lake}_{index_name}_series.csv,
    extrayendo las columnas "date" y "{index_name}_mean". Convierte los valores
    no numéricos o vacíos a np.nan para facilitar análisis numéricos posteriores.
    """

    path = ANLYSIS / f"{lake}_{index_name}_series.csv"
    dates, vals = [], []
    with path.open("r", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            d = row["date"].strip()
            v = row[f"{index_name}_mean"].strip()
            dates.append(d)
            vals.append(float(v) if v not in ("", "nan", "NaN") else np.nan)
    return dates, np.array(vals, dtype=np.float32)

In [226]:
def alignByDate(dates_c, vals_c, dates_x, vals_x):
    """
    Busca las fechas que están presentes en ambas series (dates_c y dates_x) y devuelve
    los índices en dates_c donde hay coincidencias, junto con los valores correspondientes
    en vals_x para esas fechas. Esto permite comparar o alinear dos series temporales
    basándose en fechas comunes.
    """

    idx = []
    xvals = []
    for i, d in enumerate(dates_c):
        if d in dates_x:
            j = dates_x.index(d)
            idx.append(i)
            xvals.append(vals_x[j])
    return np.array(idx, int), np.array(xvals, dtype=np.float32)

In [227]:
def corrStats(y, x):
    """
    Esta función primero filtra los valores finitos en ambos arrays para evitar cálculos
    con NaN o infinitos. Si hay menos de 3 pares válidos, devuelve NaN para las correlaciones.
    Calcula la correlación de Pearson con np.corrcoef y la correlación de Spearman
    mediante un método manual de cálculo de rangos, sin usar librerías externas.
    """

    mask = np.isfinite(y) & np.isfinite(x)
    if mask.sum() < 3:
        return {"pearson_r": np.nan, "spearman_r": np.nan, "n": int(mask.sum())}
    y2 = y[mask]; x2 = x[mask]
    pear = np.corrcoef(x2, y2)[0,1]
    # spearman (sin scipy): rank corr
    rx = x2.argsort().argsort().astype(np.float32)
    ry = y2.argsort().argsort().astype(np.float32)
    spear = np.corrcoef(rx, ry)[0,1]
    return {"pearson_r": float(pear), "spearman_r": float(spear), "n": int(mask.sum())}

In [228]:
def scatterPlot(lake, xname, x, y, dates):
    """
    La función filtra los valores finitos de x e y, luego crea un scatter plot,
    configura etiquetas y título, y finalmente guarda el gráfico en disco.
    """
    
    mask = np.isfinite(x) & np.isfinite(y)
    x2, y2 = x[mask], y[mask]
    plt.figure(figsize=(5,4))
    plt.scatter(x2, y2, s=18)
    plt.xlabel(xname)
    plt.ylabel("CYANO")
    plt.title(f"{lake}: CYANO vs {xname}")
    plt.tight_layout()
    out = OUT_REPORT / f"{lake}_scatter_{xname}.png"
    plt.savefig(out, dpi=150)
    plt.close()
    return out

In [229]:
def loadPeaks(lake):
    """
    La función intenta leer un archivo JSON con información sobre los picos
    de concentración de clorofila para un lago específico. Si el archivo no existe,
    retorna un diccionario con listas vacías.
    """
    
    p = ANLYSIS / f"{lake}_CYANO_peaks.json"
    if not p.exists():
        return {"peak_dates": [], "peak_values": []}
    return json.loads(p.read_text(encoding="utf-8"))

In [230]:
def comparePeaks():
    """
    Compara los picos de clorofila entre los lagos Atitlan y Amatitlan y guarda un resumen.

    Funcionalidad:
        - Carga los datos de picos para los lagos "Atitlan" y "Amatitlan" usando la función `loadPeaks`.
        - Calcula para cada lago:
            - El número total de picos detectados.
            - El valor máximo entre los picos (o NaN si no hay datos).
        - Guarda un archivo JSON con el resumen de estos datos en la ruta `OUT_REPORT / "peaks_summary.json"`.
    """

    a = loadPeaks("Atitlan"); b = loadPeaks("Amatitlan")
    rep = {
        "Atitlan": {"n_peaks": len(a["peak_dates"]), "max_peak": float(np.nanmax(a["peak_values"])) if a["peak_values"] else np.nan},
        "Amatitlan": {"n_peaks": len(b["peak_dates"]), "max_peak": float(np.nanmax(b["peak_values"])) if b["peak_values"] else np.nan},
    }
    (OUT_REPORT / "peaks_summary.json").write_text(json.dumps(rep, indent=2, ensure_ascii=False), encoding="utf-8")
    print("[PEAKS] resumen ->", rep)

In [231]:
def executeSection6():
    """
    Ejecuta el análisis y reporte de correlaciones entre índices remotos y la clorofila (CYANO) para varios lagos.
    
    1. Para cada lago en LAKES_N:
        - Carga las series temporales de índices: CYANO, NDVI y NDWI desde archivos CSV.
        - Alinea las series de NDVI y NDWI con la serie de CYANO, conservando solo fechas comunes.
        - Calcula las correlaciones estadísticamente relevantes (Pearson y Spearman) entre CYANO y NDVI, y entre CYANO y NDWI.
        - Genera gráficos de dispersión (scatter plots) para cada par de variables alineadas.
        - Imprime en consola la ruta de los gráficos generados.

    2. Guarda un resumen JSON con las correlaciones calculadas para cada lago en el archivo:
        - `correlations_summary.json` dentro de OUT_REPORT.

    3. Llama a `comparePeaks()` para comparar los picos de clorofila entre lagos y guardar un resumen.

    4. Imprime mensajes de estado para seguimiento de ejecución.

    Archivos generados:
    - Gráficos PNG de dispersión para cada lago y par de índices.
    - `correlations_summary.json` con las métricas de correlación.
    - `peaks_summary.json` generado por `comparePeaks`.

    """

    summary = {}
    for lake in LAKES_N:
        # Series
        dC, yC = loadSeries(lake, "CYANO")
        dN, xN = loadSeries(lake, "NDVI")
        dW, xW = loadSeries(lake, "NDWI")

        # Alinear por fecha
        idxN, xN2 = alignByDate(dC, yC, dN, xN)
        yN = yC[idxN]
        idxW, xW2 = alignByDate(dC, yC, dW, xW)
        yW = yC[idxW]

        # Correlaciones
        c_ndvi = corrStats(yN, xN2)
        c_ndwi = corrStats(yW, xW2)
        summary[lake] = {"cyano_vs_ndvi": c_ndvi, "cyano_vs_ndwi": c_ndwi}

        # Gráficos
        s1 = scatterPlot(lake, "NDVI", xN2, yN, [dC[i] for i in idxN])
        s2 = scatterPlot(lake, "NDWI", xW2, yW, [dC[i] for i in idxW])
        print(f"[OK] {lake} ->", s1.name, s2.name)

    # Guardar métricas
    (OUT_REPORT / "correlations_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    print("[OK] correlaciones -> report_outputs/correlations_summary.json")

    # Comparar picos
    comparePeaks()
    print("[DONE] p9 listo en report_outputs/")

In [232]:
if DO_PIPELINE:
    executeSection6()

[OK] Atitlan -> Atitlan_scatter_NDVI.png Atitlan_scatter_NDWI.png
[OK] Amatitlan -> Amatitlan_scatter_NDVI.png Amatitlan_scatter_NDWI.png
[OK] correlaciones -> report_outputs/correlations_summary.json
[PEAKS] resumen -> {'Atitlan': {'n_peaks': 3, 'max_peak': 170.227783203125}, 'Amatitlan': {'n_peaks': 2, 'max_peak': 167.33328247070312}}
[DONE] p9 listo en report_outputs/
